# Deceptive Points — AICC Round 0

**The task.** `train.csv` holds 180 points, but they do not come from one process. A *teacher*
produced some of them from a fixed linear law; the rest are *decoys* drawn from a different
line. The test set is teacher-only, and the score is MSE against the teacher's targets — so
fitting the training data well is exactly the wrong objective. We must recover the teacher's
law and discard the decoys.

**Approach.**

1. Show the baseline OLS fit is not merely weak but *structurally* wrong — its residuals are bimodal.
2. Model the data as a **mixture of linear regressions** and fit it with EM. BIC picks K = 2.
3. Decide **which of the two recovered lines is the teacher's**, from evidence rather than a coin flip.
4. Check the recovered law against held-out data before submitting.

**Result.** The teacher's law is recovered exactly:

$$y = 5.0 + 1.2\,x_1 - 1.0\,x_2 + 0.6\,x_3 - 0.4\,x_4 + \varepsilon,\qquad \varepsilon\sim\mathcal N(0,1^2)$$

Cross-validated MSE on held-out teacher points drops from **64.3** (baseline OLS) to **1.04**,
which is the irreducible noise floor $\sigma^2$. There is nothing left to win.

In [ ]:
import kagglehub

path = kagglehub.competition_download('deceptive-points-aicc-round-0')
print("Competition files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.linear_model import (LinearRegression, RANSACRegressor,
                                  HuberRegressor, TheilSenRegressor)

DATA_PATH = Path(path)
SUB_PATH = Path("submission.csv")
FEATURES = ["feature1", "feature2", "feature3", "feature4"]
GRID = 0.1          # resolution of the coefficient grid we test for "roundness"
RESTARTS = 40       # random restarts per EM fit

train_df = pd.read_csv(DATA_PATH / "train.csv")
test_df = pd.read_csv(DATA_PATH / "test.csv")

X, y = train_df[FEATURES].values, train_df["target"].values
X_test, test_ids = test_df[FEATURES].values, test_df["ID"].values
n = len(y)
print(f"train {X.shape}   test {X_test.shape}")

# --- plot theme: one accessible categorical pair, recessive grid and axes ---
INK, INK2, MUTED, GRIDC, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SURFACE, C1, C2, C3 = "#fcfcfb", "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.8,
    "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "legend.frameon": False, "figure.dpi": 110, "lines.linewidth": 2,
})

## 1. The baseline is not just weak — it is fitting a chimera

A single least-squares plane through all 180 points explains ~5% of the variance. That alone
would suggest "the features are uninformative". The residual histogram says otherwise: it is
**bimodal**, which no amount of noise on a single linear model can produce. Two populations are
being averaged together, and OLS is landing in the empty space between them.

In [ ]:
ols = LinearRegression().fit(X, y)
print(f"OLS R² = {ols.score(X, y):.3f}")
print("OLS coefficients:", np.round(np.r_[ols.intercept_, ols.coef_], 3))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
pred = ols.predict(X)
ax[0].scatter(pred, y, s=26, color=C1, alpha=.75, edgecolor=SURFACE, linewidth=.6)
lim = [min(y.min(), pred.min()) - 1, max(y.max(), pred.max()) + 1]
ax[0].plot(lim, lim, color=AXIS, ls="--", lw=1.5)
ax[0].set(xlabel="OLS prediction", ylabel="target",
          title=f"Baseline OLS explains almost nothing (R² = {ols.score(X, y):.3f})")

ax[1].hist(y - pred, bins=30, color=C1, edgecolor=SURFACE)
ax[1].set(xlabel="residual", ylabel="count",
          title="…and its residuals are bimodal, not Gaussian")
fig.tight_layout()
plt.show()

## 2. A mixture of linear regressions

Model each point as coming from one of $K$ linear laws:

$$p(y \mid x) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}\!\left(y \mid \beta_k^\top [1, x],\ \sigma_k^2\right)$$

EM alternates between soft-assigning points to components (E step) and refitting each component
by **weighted** least squares (M step).

Two implementation details matter:

- **Variance floor.** The likelihood of a Gaussian mixture is unbounded: a component that
  captures $p{+}1$ points can drive $\sigma_k \to 0$ and the log-likelihood to $+\infty$. Without a
  floor, BIC selects K = 4 with a spurious $\sigma = 0$ component. Flooring $\sigma_k$ at
  `0.05·std(y)` removes the degeneracy and leaves the genuine K = 2 fit untouched (its components
  sit at 0.88 and 1.02, far above the floor).
- **Random restarts.** EM is only locally optimal, so we keep the best of many Dirichlet-random
  initialisations.

In [ ]:
VAR_FLOOR = 0.05 * y.std()   # guards against the unbounded-likelihood degeneracy


def fit_mixture(X, y, K, seed=0, iters=500, tol=1e-9, var_floor=VAR_FLOOR):
    """EM for a mixture of K linear regressions. Returns coefficients, scales and responsibilities."""
    A = np.c_[np.ones(len(y)), X]
    R = np.random.default_rng(seed).dirichlet(np.ones(K), size=len(y))
    history, prev = [], -np.inf

    for _ in range(iters):
        # --- M step: weighted least squares per component ---
        B, S, pi = [], [], []
        for k in range(K):
            w = R[:, k] + 1e-12
            beta = np.linalg.solve(A.T @ (A * w[:, None]) + 1e-9 * np.eye(A.shape[1]), A.T @ (w * y))
            resid = y - A @ beta
            B.append(beta)
            S.append(max((w * resid ** 2).sum() / w.sum(), var_floor ** 2))
            pi.append(w.sum() / len(y))
        B, S, pi = np.array(B), np.array(S), np.array(pi)

        # --- E step: responsibilities, in log space ---
        logp = np.array([np.log(pi[k]) - 0.5 * np.log(2 * np.pi * S[k])
                         - (y - A @ B[k]) ** 2 / (2 * S[k]) for k in range(K)]).T
        m = logp.max(1, keepdims=True)
        R = np.exp(logp - m)
        R /= R.sum(1, keepdims=True)

        loglik = float((m.ravel() + np.log(np.exp(logp - m).sum(1))).sum())
        history.append(loglik)
        if loglik - prev < tol:
            break
        prev = loglik

    return dict(loglik=history[-1], coef=B, sigma=np.sqrt(S), weight=pi,
                resp=R, history=history, K=K)


def best_mixture(X, y, K, restarts=RESTARTS):
    """Best of `restarts` random initialisations."""
    return max((fit_mixture(X, y, K, seed=s) for s in range(restarts)), key=lambda f: f["loglik"])

In [ ]:
fits, rows = {}, []
for K in range(1, 5):
    f = fits[K] = best_mixture(X, y, K)
    n_params = K * (X.shape[1] + 2) - 1          # K·(slopes + intercept + sigma) + (K-1) weights
    rows.append(dict(K=K, loglik=f["loglik"], n_params=n_params,
                     BIC=-2 * f["loglik"] + n_params * np.log(n)))

selection = pd.DataFrame(rows)
K_best = int(selection.loc[selection.BIC.idxmin(), "K"])
display(selection.style.format({"loglik": "{:.1f}", "BIC": "{:.1f}"}).hide(axis="index"))

agreement = sum(abs(fit_mixture(X, y, 2, seed=s)["loglik"] - fits[2]["loglik"]) < 1e-3
                for s in range(RESTARTS))
print(f"K* = {K_best}    ({agreement}/{RESTARTS} restarts reach the same K=2 optimum)")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(selection.K, selection.BIC, marker="o", color=C1, ms=8)
ax[0].scatter([K_best], [selection.BIC.min()], s=180, facecolor="none",
              edgecolor=C2, linewidth=2.5, zorder=5)
ax[0].annotate(f"K* = {K_best}", (K_best, selection.BIC.min()), textcoords="offset points",
               xytext=(12, 10), color=C2, fontweight="bold")
ax[0].set(xlabel="components K", ylabel="BIC (lower is better)",
          xticks=selection.K, title="Model selection")

for K, c in zip([2, 3, 4], [C1, C2, C3]):
    ax[1].plot(fits[K]["history"], label=f"K = {K}", color=c)
ax[1].set(xlabel="EM iteration", ylabel="log-likelihood", title="EM convergence")
ax[1].legend()
fig.tight_layout()
plt.show()

## 3. Which of the two lines is the teacher's?

EM hands back two components but no labels, and the choice is worth roughly 60 MSE — so it needs
an argument, not a guess. Three independent observations point the same way.

**(a) One set of coefficients is round; the other is not.** Fit both components and compare each
coefficient to the nearest multiple of 0.1, measured in standard errors. Component 1 lands on
`[5.0, 1.2, -1.0, 0.6, -0.4]` with a worst deviation of 0.80σ (χ² p ≈ 0.90 — an excellent fit to
exactly round values). Component 0 cannot be snapped: its worst coefficient is 2.0σ off. Hand-picked
generator weights are round; a derived decoy line is not.

**(b) The decoy is a scalar multiple of the teacher.** Its slopes are ≈ −0.74× the teacher's,
i.e. the decoy lies along the *same direction in feature space*, reflected and rescaled. That is
what a decoy constructed *from* a teacher looks like, not the other way round.

**(c) The teacher's noise is exactly σ = 1.** The round component has residual scale 1.02
(standard error 0.07); the other has 0.88. A generator writing `noise=1.0` is far more likely than
one writing `noise=0.88`.

The identification rule below uses (a), since it is the only one of the three that is a
*computable criterion* rather than a post-hoc observation. §4 checks how often it is right.

In [ ]:
def coef_stderr(X, y, w, sigma2):
    """Standard errors of a weighted least-squares fit."""
    A = np.c_[np.ones(len(y)), X]
    return np.sqrt(np.diag(sigma2 * np.linalg.inv(A.T @ (A * w[:, None]))))


def roundness(fit, X, y, step=GRID):
    """For each component, how many standard errors its coefficients sit from the nearest grid point."""
    out = []
    for k in range(fit["K"]):
        beta = fit["coef"][k]
        se = coef_stderr(X, y, fit["resp"][:, k], fit["sigma"][k] ** 2)
        snapped = np.round(beta / step) * step
        z = (beta - snapped) / se
        out.append(dict(k=k, coef=beta, se=se, snapped=snapped, z=z,
                        max_abs_z=float(np.abs(z).max()),
                        p=float(stats.chi2.sf((z ** 2).sum(), len(beta))),
                        sigma=fit["sigma"][k], weight=fit["weight"][k]))
    return out


def identify_teacher(fit, X, y, step=GRID):
    """The teacher is the component whose coefficients sit closest to round values."""
    return int(np.argmin([r["max_abs_z"] for r in roundness(fit, X, y, step)]))


fit = fits[2]
report = roundness(fit, X, y)
teacher = identify_teacher(fit, X, y)
decoy = 1 - teacher

evidence = pd.DataFrame([{
    "component": r["k"],
    "mixture weight": round(r["weight"], 3),
    "σ": round(r["sigma"], 3),
    "worst |z| to 0.1-grid": round(r["max_abs_z"], 2),
    "χ² p-value": round(r["p"], 3),
    "snapped coefficients": np.array2string(r["snapped"], precision=2, suppress_small=True),
} for r in report])
display(evidence.style.hide(axis="index"))

W = report[teacher]["snapped"]
print(f"teacher = component {teacher}"
      f"  (also the majority component: {'yes' if teacher == np.argmax(fit['weight']) else 'no'})")
print(f"teacher law : y = {W[0]:+.1f} " + " ".join(f"{c:+.1f}*x{i+1}" for i, c in enumerate(W[1:]))
      + f"  +  N(0, {report[teacher]['sigma']:.2f}^2)")
print("decoy / teacher slope ratio:", np.round(fit["coef"][decoy][1:] / fit["coef"][teacher][1:], 3))

In [ ]:
labels = fit["resp"].argmax(1)
is_teacher = labels == teacher

fig, ax = plt.subplots(1, 3, figsize=(14.5, 3.9))

ax[0].hist(fit["resp"][:, teacher], bins=30, color=C1, edgecolor=SURFACE)
ax[0].set(xlabel="P(teacher | point)", ylabel="count", title="Membership is near-certain")

for k, c, name in [(teacher, C1, "teacher"), (decoy, C2, "decoy")]:
    m = labels == k
    proj = X @ fit["coef"][k][1:]
    ax[1].scatter(proj[m], y[m], s=26, color=c, alpha=.8,
                  edgecolor=SURFACE, linewidth=.6, label=name)
    span = np.array([proj[m].min(), proj[m].max()])
    ax[1].plot(span, span + fit["coef"][k][0], color=c, lw=2)
ax[1].set(xlabel="projection onto that component's own slope vector", ylabel="target",
          title="Two tight, nearly noiseless lines")
ax[1].legend()

xp = np.arange(5)
ax[2].bar(xp - .18, report[teacher]["coef"], yerr=1.96 * report[teacher]["se"],
          width=.36, color=C1, capsize=3, ecolor=INK2, label="teacher")
ax[2].bar(xp + .18, report[decoy]["coef"], yerr=1.96 * report[decoy]["se"],
          width=.36, color=C2, capsize=3, ecolor=INK2, label="decoy")
for i, (v, b, se) in enumerate(zip(report[teacher]["snapped"], report[teacher]["coef"],
                                   report[teacher]["se"])):
    ax[2].plot([i - .42, i + .06], [v, v], color=INK, lw=2, ls=(0, (1, 1)),
               label="round value" if i == 0 else None)
    sign = 1 if b >= 0 else -1
    ax[2].annotate(f"{v:g}", (i - .18, b + sign * 1.96 * se), textcoords="offset points",
                   xytext=(0, 7 * sign), ha="center", va="bottom" if sign > 0 else "top",
                   fontsize=8.5, color=INK)
ax[2].axhline(0, color=AXIS, lw=1)
ax[2].set(xticks=xp, xticklabels=["intercept", "w₁", "w₂", "w₃", "w₄"], ylabel="coefficient",
          ylim=(-1.9, 7.0), title="Only the teacher lands on round values")
ax[2].legend(loc="upper right", fontsize=9, ncol=3, columnspacing=1.0)

fig.tight_layout()
plt.show()

## 4. Sanity checks

Four things worth confirming before trusting the answer:

- The two clusters are **indistinguishable in feature space** — the split is purely in $y$. So no
  feature-based filter could have found the decoys, and conversely the teacher law extrapolates
  over the whole feature domain.
- Train and test features share the same distribution, so a law fitted on train transfers.
- The teacher's residuals are **Gaussian**, confirming we have isolated a clean generative
  component rather than carving an arbitrary subset out of a cloud.
- The identification rule from §3 is **stable under resampling** — it is not an artefact of these
  particular 180 points.

In [ ]:
A = np.c_[np.ones(n), X]
res_teacher = y[is_teacher] - A[is_teacher] @ fit["coef"][teacher]
sd = res_teacher.std(ddof=5)
print(f"teacher residuals: sd = {sd:.3f}, Shapiro-Wilk p = {stats.shapiro(res_teacher).pvalue:.3f}")

# Does the identification rule survive being shown only part of the data?
rng, hits, TRIALS = np.random.default_rng(0), 0, 25
for _ in range(TRIALS):
    idx = rng.permutation(n)[:int(.8 * n)]
    sub = best_mixture(X[idx], y[idx], 2, restarts=25)
    truth = int(np.argmin([np.abs(sub["coef"][k] - fit["coef"][teacher]).max() for k in range(2)]))
    hits += identify_teacher(sub, X[idx], y[idx]) == truth
print(f"identification rule correct on {hits}/{TRIALS} random 80% subsamples")

pca = PCA(2).fit(np.vstack([X, X_test]))
P, P_test = pca.transform(X), pca.transform(X_test)

fig, ax = plt.subplots(1, 3, figsize=(14.5, 3.9))
for k, c, name in [(teacher, C1, "teacher"), (decoy, C2, "decoy")]:
    m = labels == k
    ax[0].scatter(P[m, 0], P[m, 1], s=26, color=c, alpha=.8,
                  edgecolor=SURFACE, linewidth=.6, label=name)
ax[0].set(xlabel="PC1", ylabel="PC2",
          title="Clusters overlap in feature space\n(the deception lives in y, not x)")
ax[0].legend()

ax[1].scatter(P[:, 0], P[:, 1], s=26, color=C1, alpha=.7,
              edgecolor=SURFACE, linewidth=.6, label="train")
ax[1].scatter(P_test[:, 0], P_test[:, 1], s=26, color=C2, alpha=.7,
              edgecolor=SURFACE, linewidth=.6, label="test")
ax[1].set(xlabel="PC1", ylabel="PC2", title="Train and test features share a distribution")
ax[1].legend()

order = np.sort(res_teacher) / sd
q = (np.arange(len(order)) + .5) / len(order)
ax[2].scatter(stats.norm.ppf(q), order, s=26, color=C1, edgecolor=SURFACE, linewidth=.6)
ax[2].plot([-3, 3], [-3, 3], color=AXIS, ls="--", lw=1.5)
ax[2].set(xlabel="theoretical quantile", ylabel="standardised residual",
          title=f"Teacher residuals are N(0, {sd:.2f}²)")

fig.tight_layout()
plt.show()

## 5. Benchmark against the robust-regression alternatives

The obvious reflex for "some points are corrupted" is a robust regressor. That reflex fails here,
and the reason is worth stating: robust methods assume outliers are a *diffuse minority* around one
dominant trend. Here the decoys are 47% of the data and lie on their own tight line, so Huber and
Theil-Sen get dragged between the two. RANSAC has the right shape of assumption — find a consensus
set — but with only ~144 points per fold it does not reliably lock onto the teacher's line rather
than the decoy's, which is exactly what its high CV error reflects.

Each method is scored the way the competition scores: **MSE on held-out points that belong to the
teacher**, under 5-fold CV. The lower bound is the teacher's own noise, $\sigma^2 \approx 1.04$.

In [ ]:
class LinearLaw:
    """Wraps a fixed coefficient vector in the sklearn predict interface."""
    def __init__(self, coef):
        self.coef = np.asarray(coef, float)

    def predict(self, Z):
        return np.c_[np.ones(len(Z)), Z] @ self.coef


def em_teacher_law(X, y, snap=True, restarts=25):
    """Full pipeline as a fittable method: EM -> identify teacher -> (optionally) snap to grid."""
    f = best_mixture(X, y, 2, restarts)
    beta = f["coef"][identify_teacher(f, X, y)]
    return LinearLaw(np.round(beta / GRID) * GRID if snap else beta)


methods = {
    "OLS (baseline)": lambda a, b: LinearRegression().fit(a, b),
    "Huber": lambda a, b: HuberRegressor(max_iter=3000).fit(a, b),
    "Theil-Sen": lambda a, b: TheilSenRegressor(random_state=0, max_subpopulation=20000).fit(a, b),
    "RANSAC": lambda a, b: RANSACRegressor(LinearRegression(), random_state=0).fit(a, b),
    "EM mixture": lambda a, b: em_teacher_law(a, b, snap=False),
    "EM + snap to grid": lambda a, b: em_teacher_law(a, b, snap=True),
}

squared = {m: [] for m in methods}
for tr_idx, te_idx in KFold(5, shuffle=True, random_state=0).split(X):
    scored = te_idx[is_teacher[te_idx]]          # score on teacher points only, as the LB does
    for name, build in methods.items():
        squared[name].append((build(X[tr_idx], y[tr_idx]).predict(X[scored]) - y[scored]) ** 2)

cv = (pd.DataFrame([dict(method=m, cv_mse=float(np.concatenate(v).mean()))
                    for m, v in squared.items()])
      .sort_values("cv_mse").reset_index(drop=True))
noise_floor = report[teacher]["sigma"] ** 2
display(cv.style.format({"cv_mse": "{:.2f}"}).hide(axis="index"))
print(f"irreducible noise floor (teacher variance) = {noise_floor:.2f}")

In [ ]:
y_pred = LinearLaw(W).predict(X_test)

fig, ax = plt.subplots(1, 2, figsize=(12.5, 4))

ranked = cv.sort_values("cv_mse", ascending=False)
ax[0].barh(np.arange(len(ranked)), ranked.cv_mse.values, height=.62,
           color=[C2 if m.startswith("EM") else C1 for m in ranked.method])
ax[0].axvline(noise_floor, color=INK, ls=":", lw=1.6)
ax[0].text(noise_floor * 1.15, len(ranked) - .25, f"noise floor σ² = {noise_floor:.2f}",
           color=INK, fontsize=9, va="center")
for i, v in enumerate(ranked.cv_mse.values):
    ax[0].text(v * 1.12, i, f"{v:.2f}", va="center", fontsize=9, color=INK2)
ax[0].set(yticks=np.arange(len(ranked)), yticklabels=ranked.method.values, xscale="log",
          xlim=(0.5, 400), ylim=(-0.6, len(ranked) - 0.1),
          xlabel="5-fold CV MSE on held-out teacher points (log scale)",
          title="Only the mixture model reaches the noise floor")
ax[0].grid(axis="y", visible=False)

ax[1].hist(y[is_teacher], bins=25, color=C1, alpha=.75, edgecolor=SURFACE,
           density=True, label="train (teacher points)")
ax[1].hist(y_pred, bins=25, color=C2, alpha=.65, edgecolor=SURFACE,
           density=True, label="test predictions")
ax[1].set(xlabel="target", ylabel="density", title="Predictions land where teacher targets live")
ax[1].legend()

fig.tight_layout()
plt.show()

## 6. Submission

Predictions use the **snapped** coefficients. The snapped and unsnapped laws sit within one
standard error of each other, so this is a low-risk choice either way — but if the generator
really did use round weights (§3a puts the χ² p-value at 0.90), snapping removes the remaining
estimation error rather than baking it into every prediction. Set `SNAP = False` to submit the
raw EM estimates instead.

In [ ]:
SNAP = True

coef_final = W if SNAP else fit["coef"][teacher]
y_pred = LinearLaw(coef_final).predict(X_test)

submission_df = pd.DataFrame({"ID": test_ids, "Target": y_pred})
submission_df.to_csv(SUB_PATH, index=False)

print("law used:", np.round(coef_final, 4))
print(f"disagreement with the unsnapped fit: RMS {np.std(y_pred - LinearLaw(fit['coef'][teacher]).predict(X_test)):.4f}")
print(f"saved {len(submission_df)} rows to {SUB_PATH.resolve()}")
display(submission_df.head())

## What made this solvable

The dataset's one exploitable weakness is that the decoys are *too clean*. Had they been scattered
noise, only the teacher's line would have been tight and any robust regressor would have found it.
Instead both populations sit on near-noiseless lines, which is what makes robust regression fail —
and simultaneously what makes a two-component mixture fit so decisively (BIC gap of 530 over
K = 1, the same optimum from all 40 restarts).

That leaves the one genuinely under-determined step: EM recovers two lines but cannot say which is
the teacher's. Nothing in the likelihood distinguishes them, so the answer has to come from outside
the model — here, the observation that generator weights are written by a human and land on round
numbers, while the decoy line is a rescaling (−0.74×) of the teacher's and inherits no such
structure. Worth remembering that this last step is an inference about *how the data was made*, not
a result the data proves on its own; §4's resampling check is what keeps it honest.